# Gravando o resultado: overwrite, append e Delta Lake

## Onde estamos

Antes do pedido novo, um olhar para trás. Cada aula entregou uma parte do projeto, e cada parte tem dono e endereço:

| Aula | O que entregamos | Para quem | Onde está |
|---|---|---|---|
| 1 | o ambiente e o contrato com o fornecedor (os schemas) | o time de dados | o container e `src/wc_pipeline/schemas.py` |
| 2 | a limpeza das partidas e as estatísticas por Copa e por seleção | a transmissão | `src/wc_pipeline/transformacoes.py` |
| 3 | as consultas em SQL, o ranking por Copa e o resumo como tabela | analytics | `sql/gols_por_copa.sql` e a tabela `copa.resumo_selecoes` |
| 4 | os passes de 2018 e 2022 | analytics | `data/out/passes_por_partida` e `data/out/passes_por_copa` |

## Demanda da aula 5

O fornecedor mandou a Copa de **2026**, na pasta `data/raw_2026`, no mesmo formato do histórico. O time de dados da
Arena Analytics pediu duas coisas:

1. **As partidas e o resumo por seleção consolidados**: o histórico mais 2026, num lugar só, de onde a transmissão e o
   time de analytics leiam direto. As funções da aula 2 continuam fazendo a conta; o que muda é que o resultado passa a
   ficar gravado, sempre atualizado.
2. **Que isso aconteça sozinho a cada Copa nova.** O fornecedor vai mandar uma pasta assim a cada Copa, e o time não
   quer depender de alguém abrir um notebook. A carga das partidas precisa virar um **job**: um programa que roda com
   um comando, que um agendador pode chamar, e que pode rodar de novo sem estragar nada.

Por que só essa parte? Porque é a única **recorrente**. Os passes da aula 4 foram um estudo pontual sobre 2018 e 2022,
que só se refaz se alguém pedir. As partidas e o resumo são a base de toda transmissão, de toda Copa.

| Etapa | O que fazer | Notebook |
|---|---|---|
| Task 1 | Definir como e em que formato gravar as tabelas consolidadas | `01_overwrite_e_delta` |
| Task 2 | Organizar as tabelas por Copa | `02_partitionby` |
| Task 3 | Controlar os arquivos gravados e carregar a Copa de 2026 sem apagar o histórico | `03_repartition_e_overwrite_dinamico` |
| Task 4 | Transformar a carga num job, que roda com um comando a cada Copa nova, e carregar 2026 | `04_locais_e_pipeline_final` |

**Pronto quando:** as tabelas de partidas e de resumo por seleção têm as 23 Copas; são gravadas num formato que
protege contra falhas e guarda o histórico de versões, organizado por Copa, com um arquivo por pasta; e são carregadas
por um comando que recebe a pasta da Copa nova, sem apagar nem duplicar o histórico, mesmo se rodar duas vezes, com a
saída conferida automaticamente.

## Task 1: definir como e em que formato gravar as tabelas consolidadas

Esta sessão tem três linhas a mais que as de sempre. Elas preparam o **Delta Lake**, um formato que você ainda não
conhece: ele aparece mais adiante neste vídeo, e aí as três linhas são explicadas. A célula já vem pronta.

In [1]:
import sys
sys.path.insert(0, "../../src")

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession, functions as F

construtor = (
    SparkSession.builder
    .appName("arena-escrita")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(construtor).getOrCreate()

RAW = "../../data/raw"
OUT = "../../data/out"

spark

## As partidas do projeto

`ler_partidas(pasta)`: a mesma leitura serve para o histórico e para a Copa de 2026, que chega em outra pasta.

In [2]:
from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas

def ler_partidas(pasta):
    return limpar_partidas(
        spark.read.option("header", "true").schema(MATCHES).csv(f"{pasta}/WorldCupMatches.csv")
    )

historico = ler_partidas(RAW)

In [3]:
historico.count()

964

A próxima célula já vem pronta: ela só apaga as pastas que este notebook grava, para ele poder rodar de novo.

In [4]:
import shutil
for pasta in ["partidas_parquet", "partidas_delta"]:
    shutil.rmtree(f"{OUT}/{pasta}", ignore_errors=True)

## E se o job rodar de novo?

| Modo | Se o destino já existe |
|---|---|
| `append` | acrescenta arquivos novos |
| `overwrite` | apaga e grava de novo |

Sem modo nenhum, o padrão é dar **erro** se a pasta já existe: o projeto sempre declara o modo.

In [5]:
historico.write.mode("overwrite").parquet(f"{OUT}/partidas_parquet")
spark.read.parquet(f"{OUT}/partidas_parquet").count()

964

In [6]:
historico.write.mode("append").parquet(f"{OUT}/partidas_parquet")
spark.read.parquet(f"{OUT}/partidas_parquet").count()

1928

In [7]:
historico.write.mode("overwrite").parquet(f"{OUT}/partidas_parquet")
spark.read.parquet(f"{OUT}/partidas_parquet").count()

964

**`append` num job que pode rodar de novo duplica o dado, sem erro.** O modo do projeto é o `overwrite`.

## O risco do overwrite

E se o job rodar com um filtro de teste esquecido?

In [8]:
historico.filter(F.col("ano") >= 2010).write.mode("overwrite").parquet(f"{OUT}/partidas_parquet")
spark.read.parquet(f"{OUT}/partidas_parquet").count()

256

O histórico antigo sumiu, e o Parquet não tem "desfazer": no `overwrite`, o dado é apagado antes de o novo ser
gravado, e a escrita não é atômica.

## Os formatos e o Delta Lake

| Formato | Como guarda |
|---|---|
| **CSV** | texto, uma linha por registro, sem tipos |
| **JSON** | texto, repete o nome de cada coluna em cada linha |
| **Parquet** | binário e colunar, com o schema (vídeo 4.4) |

O **Delta Lake** guarda o dado em **Parquet** e acrescenta a pasta `_delta_log/`, com um registro de cada escrita:

- **escrita atômica**: só vale quando é registrada no log, no fim;
- **versões**: as anteriores continuam disponíveis.

In [9]:
caminho_delta = f"{OUT}/partidas_delta"

historico.write.format("delta").mode("overwrite").save(caminho_delta)

In [10]:
!ls {caminho_delta}

_delta_log  part-00000-d12976b1-a907-4317-a55c-5f162be4503e-c000.snappy.parquet


In [11]:
!ls {caminho_delta}/_delta_log

00000000000000000000.crc  00000000000000000000.json  _staged_commits


Os `part-...parquet` estão lá; a novidade é o `_delta_log`, com o registro da **versão 0**.

## O mesmo engano, agora em Delta

In [12]:
historico.filter(F.col("ano") >= 2010).write.format("delta").mode("overwrite").save(caminho_delta)
spark.read.format("delta").load(caminho_delta).count()

256

A versão atual ficou com o engano, mas os arquivos da versão 0 continuam lá:

In [13]:
from pathlib import Path

spark.sql(f"DESCRIBE HISTORY delta.`{Path(caminho_delta).resolve()}`").select("version", "operation").show()

+-------+---------+
|version|operation|
+-------+---------+
|      1|    WRITE|
|      0|    WRITE|
+-------+---------+



In [14]:
versao_0 = spark.read.format("delta").option("versionAsOf", 0).load(caminho_delta)
versao_0.count()

964

Para desfazer, leia a versão 0 e grave de novo:

In [15]:
versao_0.write.format("delta").mode("overwrite").save(caminho_delta)

In [16]:
spark.read.format("delta").load(caminho_delta).count()

964

## A decisão do projeto

| Formato | Use para |
|---|---|
| **CSV** | trocar arquivos com pessoas e sistemas antigos |
| **JSON** | dado semiestruturado, respostas de APIs |
| **Parquet** | dado analítico gravado de uma vez |
| **Delta Lake** | tabelas de pipeline, regravadas, com transação e histórico |

**Decisão:** `overwrite`, em **Delta Lake**.

Dois problemas para os próximos vídeos: quem consulta pergunta por **uma Copa** e o Spark lê a tabela inteira; e o
`overwrite` troca **a tabela inteira**, então a Copa de 2026 apagaria as outras da versão atual.

## Resultado da task 1

`overwrite` em Delta Lake. Próxima task: organizar as tabelas por Copa.

## Para consultar depois

- **Os outros dois modos:** `error` (ou `errorifexists`), o padrão, lança erro se o destino existe; `ignore` não faz
  nada se o destino existe, sem avisar.
- **A mesma escrita, de outro jeito:** `df.write.format("parquet").mode("overwrite").save(caminho)` é o mesmo que
  `df.write.mode("overwrite").parquet(caminho)`.
- **`MERGE`:** quando o lote que chega é **parcial**, como as partidas do dia durante uma Copa, o Delta permite
  atualizar as linhas que já existem e inserir as novas numa operação só, com `MERGE INTO`. No projeto, cada lote é uma
  Copa inteira, e o vídeo 5.3 mostra o caminho mais simples para esse caso.
- **`VACUUM`:** apaga os arquivos que não fazem parte de nenhuma versão recente. Sem ele, as versões antigas ocupam
  espaço para sempre.
- **Outros formatos de tabela:** Apache Iceberg e Apache Hudi seguem a mesma ideia do Delta, com metadados sobre
  arquivos Parquet.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [17]:
spark.stop()